# 🔬 MedSymbol in Google Colab

**Setup & Training Guide**

This notebook will:
1. ✅ Mount Google Drive
2. ✅ Clone MedSymbol repository
3. ✅ Install dependencies
4. ✅ Verify environment
5. ✅ Download data
6. ✅ Run tests
7. ✅ Train/inference

---

## 1️⃣ Mount Google Drive & Clone Repository

In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')
print("✓ Google Drive mounted")

In [ ]:
# Clone repository
!cd /content && git clone https://github.com/udshah31/medsymbol.git
!cd /content/medsymbol && git pull origin master
print("✓ Repository cloned")

In [ ]:
# Check environment
import subprocess
print("Python Version:")
!python --version
print("\nGPU Available:")
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 2️⃣ Install Dependencies

In [ ]:
# Install MedSymbol with all dependencies
%cd /content/medsymbol
!pip install -q -e .
print("✓ MedSymbol installed")

In [ ]:
# Install additional tools
!pip install -q kaggle  # For dataset download
!pip install -q wandb   # For experiment tracking (optional)
print("✓ Additional packages installed")

## 3️⃣ Verify Environment

In [ ]:
# Setup Python path
import sys
sys.path.insert(0, '/content/medsymbol')

# Run verification script
%cd /content/medsymbol
!python scripts/verify_env.py

In [ ]:
# Test imports
print("Testing imports...\n")

try:
    from src.symbolic.disease_ontology import DiseaseOntology
    print("✓ disease_ontology")
except Exception as e:
    print(f"✗ disease_ontology: {e}")

try:
    from src.encoders.vision import VisionEncoder
    print("✓ vision encoder")
except Exception as e:
    print(f"✗ vision encoder: {e}")

try:
    from src.error_handling import ErrorHandler
    print("✓ error handling")
except Exception as e:
    print(f"✗ error handling: {e}")

try:
    from src.clinical_support import ClinicalRecommender
    print("✓ clinical support")
except Exception as e:
    print(f"✗ clinical support: {e}")

try:
    from src.production import AuditLogger
    print("✓ production monitoring")
except Exception as e:
    print(f"✗ production monitoring: {e}")

print("\n✓ All core modules imported successfully!")

## 4️⃣ Setup Data Directory

In [ ]:
import os
from pathlib import Path

# Create persistent data directory in Google Drive
DATA_DIR = '/content/drive/MyDrive/medsymbol_data'
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(f'{DATA_DIR}/raw/nih_cxr14', exist_ok=True)
os.makedirs(f'{DATA_DIR}/processed', exist_ok=True)

print(f"Data directory: {DATA_DIR}")
print("Contents:")
!ls -lh {DATA_DIR}/

## 5️⃣ Download Dataset (Choose Option)

### Option A: Download NIH Metadata Only (5 MB)

In [ ]:
# Download NIH CXR-14 metadata CSV
DATA_DIR = '/content/drive/MyDrive/medsymbol_data'

!cd {DATA_DIR}/raw/nih_cxr14 && \
  wget -q https://nihcc.box.com/shared/static/itsreall706qrcf160r3l7uc3llrzekk -O Data_Entry_2017.csv && \
  ls -lh

print("\n✓ NIH metadata downloaded")
print(f"✓ Location: {DATA_DIR}/raw/nih_cxr14/Data_Entry_2017.csv")

### Option B: Download via Kaggle (Full 112GB Images - Optional)

In [ ]:
# ONLY RUN IF YOU HAVE KAGGLE ACCOUNT
# 1. Get API key from https://www.kaggle.com/settings/account
# 2. Uncomment and run below:

# from google.colab import files
# print("Upload kaggle.json from https://www.kaggle.com/settings/account")
# files.upload()
# 
# !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !cd /content/drive/MyDrive/medsymbol_data/raw && kaggle datasets download -d nih-chest-xrays/chest-xray-14 --unzip
# print("✓ Dataset downloaded")

### Option C: Link Existing Data

In [ ]:
# If you already have data in Drive, symlink it
DATA_DIR = '/content/drive/MyDrive/medsymbol_data'

# Symlink to repo
!ln -sf {DATA_DIR} /content/medsymbol/data

print(f"✓ Symlinked {DATA_DIR} → /content/medsymbol/data")

## 6️⃣ Test Functionality

In [ ]:
import torch

print("System Status:")
print(f"  GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("  Using CPU")

In [ ]:
# Test VisionEncoder
from src.encoders.vision import VisionEncoder

device = 'cuda' if torch.cuda.is_available() else 'cpu'
encoder = VisionEncoder().to(device)

# Create dummy X-ray
dummy_xray = torch.randn(1, 1, 224, 224).to(device)
with torch.no_grad():
    features = encoder(dummy_xray)

print(f"✓ Vision Encoder Test:")
print(f"  Input shape: {dummy_xray.shape}")
print(f"  Output shape: {features.shape}")
print(f"  Device: {device}")

In [ ]:
# Test Clinical Recommendations
from src.clinical_support import ClinicalRecommender, DecisionSupportFormatter

# Generate explanation
explanation = ClinicalRecommender.generate_explanation(
    disease="pneumonia",
    probability=0.87,
    findings=["consolidation_RLL", "fever", "elevated_wbc"],
    severity="moderate"
)

# Format for display
formatter = DecisionSupportFormatter()
report = formatter.format_report(explanation)
print(report)

## 7️⃣ Run Tests

In [ ]:
import subprocess

%cd /content/medsymbol

result = subprocess.run(
    ["python", "-m", "pytest", "tests/test_all.py", "-v", "--tb=short"],
    capture_output=True,
    text=True,
    timeout=120
)

print(result.stdout)
if result.returncode != 0:
    print("\nErrors:")
    print(result.stderr)

## 8️⃣ Training Example (Optional)

In [ ]:
# Example: Simple training loop with dummy data
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm

# Configuration
device = 'cuda' if torch.cuda.is_available() else 'cpu'
EPOCHS = 3
BATCH_SIZE = 16
LR = 1e-4

# Create dummy data
print("Creating dummy dataset...")
X = torch.randn(100, 1, 224, 224)  # 100 X-rays
y = torch.randint(0, 14, (100,))    # 14 diseases
dataset = TensorDataset(X, y)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

# Initialize model
from src.encoders.vision import VisionEncoder
model = VisionEncoder().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=LR)

# Training loop
print(f"\nTraining on {device.upper()}...\n")

for epoch in range(EPOCHS):
    total_loss = 0
    
    for batch_idx, (x, y_true) in enumerate(tqdm(loader, desc=f"Epoch {epoch+1}/{EPOCHS}")):
        x, y_true = x.to(device), y_true.to(device)
        
        # Forward
        logits = model(x)
        loss = criterion(logits, y_true)
        
        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    avg_loss = total_loss / len(loader)
    print(f"Loss: {avg_loss:.4f}")

print("\n✓ Training complete!")

## 9️⃣ Save Checkpoints

In [ ]:
import os

# Create checkpoint directory
CHECKPOINT_DIR = '/content/drive/MyDrive/medsymbol_checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Save trained model
checkpoint_path = f'{CHECKPOINT_DIR}/vision_encoder_trained.pt'
torch.save(model.state_dict(), checkpoint_path)

print(f"✓ Checkpoint saved: {checkpoint_path}")
print(f"✓ Size: {os.path.getsize(checkpoint_path) / 1e6:.1f} MB")

## 🔟 Next Steps

In [ ]:
print("""
╔══════════════════════════════════════════════════════════╗
║           MedSymbol Setup Complete! ✓                  ║
╚══════════════════════════════════════════════════════════╝

📚 Next Steps:

1. READ DOCUMENTATION:
   - COLAB_SETUP.md (full guide)
   - README.md (architecture)
   - src/clinical_documentation.py (clinical info)

2. CUSTOMIZE FOR YOUR DATA:
   - Update data loading in scripts/download_datasets.py
   - Modify configs/default.yaml for hyperparameters
   - Create your own training script in scripts/

3. MONITOR EXPERIMENTS:
   - Use Weights & Biases (wandb) for tracking
   - Save checkpoints to Google Drive
   - Monitor GPU usage with nvidia-smi

4. RUN FULL TRAINING:
   - python scripts/train_model.py (when ready)
   - python scripts/evaluate.py (evaluate performance)
   - python scripts/inference.py (run on new data)

5. DEPLOY:
   - FastAPI server for inference
   - Docker container for production
   - Monitor with AuditLogger & DriftDetector

📞 Support:
   - GitHub: https://github.com/udshah31/medsymbol
   - Issues: https://github.com/udshah31/medsymbol/issues

🎉 Happy researching!
""")